In [0]:
acc_name = "medallionazure"
container = "data"
acc_key = dbutils.secrets.get(scope="azure-storage", key="storage-key")
spark.conf.set(f"fs.azure.account.key.{acc_name}.blob.core.windows.net", acc_key)
silver_path = f"wasbs://data@{acc_name}.blob.core.windows.net/silver/yellow_taxi"
spark.read.parquet(silver_path).createOrReplaceTempView("taxi_trips_silver")

import pandas as pd
zones_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zones_pd = pd.read_csv(zones_url)
spark.createDataFrame(zones_pd).createOrReplaceTempView("taxi_zones")

print("✅ Tabele zarejestrowane. Możesz teraz używać SQL!")

In [0]:
gold_df = spark.sql("""
    SELECT 
        z.Borough,
        z.Zone,
        COUNT(*) as total_trips,
        ROUND(SUM(t.total_amount), 2) as total_revenue,
        ROUND(AVG(t.trip_distance), 2) as avg_distance
    FROM 
        taxi_trips_silver t
    JOIN 
        taxi_zones z ON t.PULocationID = z.LocationID
    GROUP BY 
        z.Borough, z.Zone
""")

# Zapisujemy fizycznie do folderu Gold
gold_path = "wasbs://data@medallionazure.blob.core.windows.net/gold/taxi_analysis"
gold_df.write.mode("overwrite").parquet(gold_path)

print("✅ Dane zostały fizycznie zapisane w folderze GOLD!")